# 02 · Tools

In notebook 01 the model invented ShipFast's refund policy and an order status, because it cannot see
either. **Tools** fix that: functions the model can ask your code to run, whose results it then reads.

**You will learn**

- where the tools live, and why they are plain Python functions
- how CrewAI turns a function into a tool
- what a tool call is, and how to run one and hand the result back
- how tool errors and safety checks work when the model gets something wrong

In [1]:
from crew_agents.llm import make_llm  # import crew_agents first: it switches CrewAI's telemetry off

from agent_contracts.harness import verify_model

# Check that Ollama is running and the shared model exists, then build the shared CrewAI model (crew_agents/llm.py).
verify_model()
llm = make_llm()  # the same model setup as notebook 01

## The tools are plain functions

ShipFast's five tools live once, in `agent_contracts.shipfast.tools`, shared by all three tutorials. They
know nothing about CrewAI: each takes simple arguments and returns a JSON string. You can call them
directly.

In [2]:
# The shared tools live in contracts, not in CrewAI. TOOLS is the list of the five plain functions.
from agent_contracts.shipfast import tools as shipfast

shipfast.reset()  # start clean: clears the refund ledger, the tickets and the tool-call log
print("ShipFast's tools:", ", ".join(f.__name__ for f in shipfast.TOOLS))

ShipFast's tools: lookup_order, track_shipment, search_policy, issue_refund, escalate_to_human


Call one directly, like any function. `lookup_order` takes an order ID and returns a JSON string, which
the cell below parses so it prints as one field per line.

In [3]:
import json

# Call a tool by hand: the argument is an order ID, the return value is a JSON string.
order = json.loads(shipfast.lookup_order("A1001"))

for field, value in order.items():
    print(f"{field:13}{value}")

order_id     A1001
customer     Maria Santos
item         Wireless Headphones
total        89.0
status       in_transit
carrier      LBC
ordered_on   2026-10-28
eta          2026-11-04
delivered_on None


## Turning a function into a tool

CrewAI's `tool(name)` decorator wraps a function as a `BaseTool`: it reads the type hints for the
argument schema and the docstring for the description the model sees. `crew_agents/tools.py` wraps all
five:

```python
def _wrap(functions) -> list[BaseTool]:
    # tool(name) returns a decorator; applying it to f gives a CrewAI BaseTool named after the function.
    # CrewAI reads f's type hints for the argument schema and its docstring for the description the model
    # sees. Calling the tool still runs the shared function, unchanged.
    return [tool(f.__name__)(f) for f in functions]
```

In [ ]:
# The ShipFast tools, wrapped, and a helper that lists their schemas; both are used below.
from crew_agents.tools import function_schemas, shipfast_tools

Build the tools and look at what the model is told about each one: its name and its arguments, with their
types.

In [5]:
tools = shipfast_tools()  # one CrewAI tool per function

print(f"{'tool':18} arguments")
for t in tools:
    # the argument schema CrewAI built from the function's type hints
    properties = t.args_schema.model_json_schema()["properties"]
    arguments = ", ".join(f"{name}: {spec['type']}" for name, spec in properties.items())
    print(f"{t.name:18} {arguments}")

tool               arguments
lookup_order       order_id: string
track_shipment     order_id: string
search_policy      query: string
issue_refund       order_id: string, amount: number, reason: string
escalate_to_human  order_id: string, summary: string


The docstring is the only instruction the model gets about when to use a tool, so it matters as much
as the code. `issue_refund`'s docstring, for example, says the amount cannot exceed the order total.

## A tool call

Inside a CrewAI agent you never handle tool calls yourself (notebook 03). Calling the `LLM` directly
shows what happens underneath. Pass the tool descriptions in the OpenAI function format
(`function_schemas` converts them) and the model can reply with a **tool call** instead of text: the
name of a tool and the arguments to run it with. It does not run anything itself.

In [6]:
# The instructions for this notebook: check facts with the tools instead of guessing.
SPECIALIST = ("You are ShipFast's support specialist. Use the tools to check facts before you answer; "
              "never guess order details or policy.")

# function_schemas converts the tools to the OpenAI function format that llm.call accepts.
schemas = function_schemas(tools)

Now send the customer's message and look at the reply. When the model wants a tool, `call` returns a list
of tool calls instead of a string. If it
answers in text instead, which smaller models sometimes do, the cell says so.

In [7]:
messages = [{"role": "system", "content": SPECIALIST}, {"role": "user", "content": "Hi, where is my order A1001?"}]
calls = llm.call(messages, tools=schemas)  # a list of tool calls when the model asks for a tool, else a string

if isinstance(calls, str):  # the model answered straight away instead of asking for a tool
    print("no tool call: the model answered in text instead (smaller models do this more often)")
    print("answer:", calls.strip()[:200])
else:
    print("tool calls:", len(calls))
    for c in calls:
        print("  tool:     ", c.function.name)
        print("  arguments:", c.function.arguments)  # a JSON string
        print("  call id:  ", c.id)  # the tool's result must carry this id back

tool calls: 1
  tool:      track_shipment
  arguments: {"order_id":"A1001"}
  call id:   call_265zqz60


## Running the call and handing back the result

Your code runs the tool, then appends two messages in the OpenAI format: the assistant's tool call, and
a `tool` message carrying the result and the call's id, so the model can match result to request. Ask
again: now the answer is grounded in real data.

In [8]:
# Look tools up by name, so a tool call from the model can be matched to the tool to run.
tools_by_name = {t.name: t for t in tools}


def run_calls(messages, calls):
    """Run each requested tool and append the assistant and tool messages."""
    if isinstance(calls, str):  # a string is a text answer: there is no tool to run
        print("no tool to run: the model answered in text")
        return
    # first the model's own turn (its tool calls), in the OpenAI message format
    messages.append({"role": "assistant", "content": "", "tool_calls": [
        {"id": c.id, "type": "function", "function": {"name": c.function.name, "arguments": c.function.arguments}}
        for c in calls]})
    for c in calls:
        result = tools_by_name[c.function.name].run(**json.loads(c.function.arguments))
        print("tool ran:    ", c.function.name, c.function.arguments)
        print("result start:", result[:100], "...")  # the start of the JSON the tool returned
        # then a tool message with the result, carrying the call's id so the model can match them
        messages.append({"role": "tool", "tool_call_id": c.id, "content": result})

Run the tool the model asked for, then ask the model again. This time it has real data to answer from.

In [9]:
run_calls(messages, calls)
answer = llm.call(messages, tools=schemas)

print("answer:", answer)

tool ran:     track_shipment {"order_id":"A1001"}
result start: {"order_id": "A1001", "status": "in_transit", "carrier": "LBC", "eta": "2026-11-04", "delivered_on": ...


answer: Your order A1001 is currently in transit with LBC. Here's the latest update:

- **Status:** In Transit
- **Carrier:** LBC
- **Estimated Time of Arrival (ETA):** November 4, 2026

**Tracking Events:**
- October 28, 2026: Order packed at Manila warehouse
- October 30, 2026: Picked up by LBC
- October 31, 2026: Arrived at Cebu sorting hub

If you have any further questions, feel free to ask!


That is the whole idea of an agent, done by hand: call the model, run the tools it asks for, feed the
results back, repeat until it answers in text. Notebook 03 lets a CrewAI `Agent` run that loop.

CrewAI also has a one-step shortcut: pass `available_functions` and `call` runs the requested tool
itself. Note what comes back: the tool's raw output, not an answer. It runs one round, not the loop.

In [10]:
# available_functions maps tool names to the functions to run. messages[:2] is only the system and user
# message, so the model asks for the tool again and call() runs it.
shortcut = llm.call(messages[:2], tools=schemas, available_functions={t.name: t.run for t in tools})

# str: the tool's raw output (or the model's text, if it asked for no tool)
print("returned type:", type(shortcut).__name__)
print("returned text:", shortcut[:120], "...")

returned type: str
returned text: {"order_id": "A1001", "status": "in_transit", "carrier": "LBC", "eta": "2026-11-04", "delivered_on": null, "tracking": [ ...


## Which tool does it reach for?

Here is the model's **first** move on the opening message of each ShipFast scenario. Nothing is run; we
only look at what it asks for.

In [11]:
# Show the model's first move for the opening message of every ShipFast scenario. Nothing is run.
from agent_contracts.shipfast.scoring import load_scenarios

seen = set()  # opening messages already tried: two scenarios start with the same one
with_tool = 0  # how many messages started with a tool call
print(f"{'customer message':50} -> first move")
for scenario in load_scenarios():
    text = scenario["turns"][0]  # the customer's first message
    if text in seen:
        continue
    seen.add(text)
    first = llm.call([{"role": "system", "content": SPECIALIST}, {"role": "user", "content": text}], tools=schemas)
    if isinstance(first, list):  # a list means tool calls; a string means a plain text answer
        asked = ", ".join(f"{c.function.name}({c.function.arguments})" for c in first)
        with_tool += 1
    else:
        asked = "(no tool) " + first[:60]
    print(f"{text[:48]:50} -> {asked}")  # long messages are cut to 48 characters

print(f"\n{with_tool} of {len(seen)} messages started with a tool call")

customer message                                   -> first move


Hi, where is my order A1001?                       -> track_shipment({"order_id":"A1001"})


My order A1002 never arrived and tracking says i   -> track_shipment({"order_id":"A1002"})


Order A1003 is more than a week late. Please can   -> track_shipment({"order_id":"A1003"})


My vase set from order A1004 arrived broken.       -> track_shipment({"order_id":"A1004"})


I want to return order A1005, the stand wobbles.   -> lookup_order({"order_id":"A1005"})


Where is order A9999?                              -> track_shipment({"order_id":"A9999"})


Can you write me a poem about cats?                -> (no tool) I can't write poems, but I can help you find information or 

6 of 7 messages started with a tool call


Look for two things: whether each choice makes sense for the message, and the out-of-scope request,
where the right move is no tool at all. The model and settings are identical to the LangGraph tutorial,
so its choices should match the ones there; a difference would point to how the framework sends the
request.

## When a tool says no

Tools report problems as data, not exceptions, so the model can read them and recover. An unknown order
comes back as `order_not_found`, and policy P6 says to ask the customer to check the ID.

In [12]:
# The by-hand loop again, on an order that does not exist. A few rounds: ask, run the tools, ask again.
messages = [{"role": "system", "content": SPECIALIST}, {"role": "user", "content": "Where is order A9999?"}]
for _ in range(3):
    reply = llm.call(messages, tools=schemas)
    if not isinstance(reply, list):
        break  # a string: the model answered in text, so the loop is done
    run_calls(messages, reply)
print("final answer:", reply)

tool ran:     track_shipment {"order_id":"A9999"}
result start: {"error": "order_not_found", "order_id": "A9999"} ...


final answer: The order ID "A9999" could not be found. Please double-check the order ID and try again. If you need further assistance, feel free to ask!


## Safety lives in the tool, not the prompt

In one of our early test runs, the model once tried to refund $120 on a $45 order without looking up
the total. The prompt cannot prevent that; the tool can. `issue_refund` refuses any amount above the
order total, whoever calls it.

In [13]:
# Try to refund $120 on an order that cost $45 (A1002), then the correct $45. The tool judges both itself.
too_much = shipfast.issue_refund("A1002", 120, "lost package")
correct = shipfast.issue_refund("A1002", 45, "lost package")

print("refund of $120:", too_much)
print("refund of $45: ", correct)

refund of $120: {"error": "amount_exceeds_order_total", "order_id": "A1002", "amount": 120.0, "order_total": 45.0}
refund of $45:  {"refund_id": "RF-A1002", "order_id": "A1002", "amount": 45.0, "status": "issued"}


Only one of those should have reached the ledger, the record of refunds that were really issued.

In [14]:
# refunds() is the ledger: one entry per refund that was actually issued.
print("ledger:")
for r in shipfast.refunds():
    print(f"  {r['refund_id']}  order {r['order_id']}  ${r['amount']:.2f}  {r['status']}  ({r['reason']})")

ledger:
  RF-A1002  order A1002  $45.00  issued  (lost package)


The ledger is what ShipFast's test runner trusts: a result only counts as `refunded` if the ledger
has a matching refund. The model's claim is never enough.

The shared tools also record every call, whichever framework made it. Notebook 07 uses this so all three
tutorials count tool calls the same way.

In [15]:
# Every call to a shared tool is logged, whichever framework made it. Read the log, then reset it.
print("tools called, in order:", ", ".join(shipfast.tool_calls()))
shipfast.reset()  # leave a clean state behind

tools called, in order: lookup_order, track_shipment, track_shipment, track_shipment, issue_refund, issue_refund


## How the others do this

- **LangGraph** (LangChain) wraps the same functions with `tool()` and attaches them with
  `llm.bind_tools(tools)`; tool calls come back on the reply as `ai.tool_calls`, and invoking a tool with
  the call returns a ready-made `ToolMessage`.
- **Strands** also uses a `@tool` decorator, and its `Agent` runs the call-execute-repeat loop for you
  from the first example.

All three wrap the same five functions from `agent_contracts`, so the tools behave identically; only the
wrapping differs.

## Takeaways

- A tool is a plain function plus a description the model reads; keep the logic outside the framework.
- The model only *asks* for a tool. Your code (or the framework) runs it and hands back the result.
- CrewAI's `available_functions` runs one round of tools; the loop belongs to an `Agent`.
- Errors come back as data the model can act on, and safety checks belong in the tool.
- Next: **03 · Agent**, where CrewAI runs the loop for you.